In [31]:
import kfp
from kfp import dsl, compiler
from kfp.dsl import component, Input, Output, Dataset, Model, Metrics
from typing import NamedTuple
from google.cloud import aiplatform
from google.cloud.aiplatform.pipeline_jobs import PipelineJob
from google.cloud import storage
import time
from google.cloud import aiplatform
from google.cloud import storage
import pandas as pd
import numpy as np
import joblib
import os

In [32]:
project = !gcloud config get-value project
PROJECT_ID = project[0]
PROJECT_ID = "mlops-pipeline-01"
REGION = "europe-west3"
BUCKET_NAME = PROJECT_ID


MODEL_DISPLAY_NAME = "knn-model-dev"
ENDPOINT_NAME = "knn-endpoint-dev"


print("Project:", PROJECT_ID)
print("Region:", REGION)

Project: mlops-pipeline-01
Region: europe-west3


In [33]:
!pip install -q google-cloud-aiplatform scikit-learn pandas numpy joblib kfp

In [34]:
aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=BUCKET_NAME
)

In [35]:
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

dfs = []

for blob in bucket.list_blobs():
    if blob.name.endswith("_breakdowns.csv"):
        print("Reading", blob.name)
        df = pd.read_csv(blob.open("r"))
        if "BREAKS" in df.columns:
            dfs.append(df)

if not dfs:
    raise RuntimeError("No valid CSV files found")

raw_df = pd.concat(dfs, ignore_index=True)
raw_df.head()

Reading 0Hybrid_Framework_NSGA3_Detailed_schedule0_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule1_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule2_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule3_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule4_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule5_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule6_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule7_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule8_breakdowns.csv
Reading 0Hybrid_Framework_NSGA3_Detailed_schedule9_breakdowns.csv


,ID,Priority,Family_type,First_stage,Start_time_S1,Finish_time_S1,Processing_Time_S1,Second_stage,Start_time_S2,Finish_Time_S2,...,Finish_time_S3,Processing_Time_S3,Fourth_stage,Start_time_s4,Finish_time,Processing_Time_S4,Overall_processing_time,Overall_waiting_time,Tardiness,BREAKS
0,12,1,34,SMD_0,0,402,402,AOI_2,402,781,...,0,0,CC_1,781,979,198,979,0,0.0,0
1,23,1,37,SMD_1,0,354,354,AOI_0,354,441,...,592,151,CC_1,592,696,104,696,0,0.0,0
2,131,1,29,SMD_3,0,37,37,AOI_0,37,90,...,138,48,0,0,0,0,138,0,0.0,0
3,29,1,18,SMD_0,402,1155,753,AOI_2,1155,1457,...,2127,670,CC_0,2263,2461,198,1923,538,0.0,0
4,75,1,20,SMD_4,0,189,189,AOI_1,189,766,...,1021,255,0,0,0,0,1021,0,0.0,0


In [36]:
prepared_rows = []

for _, row in raw_df.iterrows():
    prepared_rows.append({
        "job_id": row.iloc[0],
        "priority": row.iloc[1],
        "smd_0": int(row.iloc[3] == "SMD_0"),
        "smd_1": int(row.iloc[3] == "SMD_1"),
        "processing_time_s1": row.iloc[6],
        "aoi_0": int(row.iloc[7] == "AOI_0"),
        "processing_time_s2": row.iloc[10],
        "ss_0": int(row.iloc[11] == "SS_0"),
        "processing_time_s3": row.iloc[14],
        "cc_0": int(row.iloc[15] == "CC_0"),
        "processing_time_s4": row.iloc[18],
        "overall_processing_time": row.iloc[19],
        "overall_waiting_time": row.iloc[20],
        "tardiness": row.iloc[21],
        "breaks": row.iloc[22],
    })

prepared_df = pd.DataFrame(prepared_rows)
prepared_df.head()

,job_id,priority,smd_0,smd_1,processing_time_s1,aoi_0,processing_time_s2,ss_0,processing_time_s3,cc_0,processing_time_s4,overall_processing_time,overall_waiting_time,tardiness,breaks
0,12,1,1,0,402,0,379,0,0,0,198,979,0,0.0,0
1,23,1,0,1,354,1,87,1,151,0,104,696,0,0.0,0
2,131,1,0,0,37,1,53,1,48,0,0,138,0,0.0,0
3,29,1,1,0,753,0,302,0,670,1,198,1923,538,0.0,0
4,75,1,0,0,189,0,577,1,255,0,0,1021,0,0.0,0


In [37]:
from sklearn.neighbors import KNeighborsClassifier

X = prepared_df.drop("breaks", axis=1)
y = prepared_df["breaks"]

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X, y)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [38]:
MODEL_DIR = "model"
MODEL_FILE = "model.joblib"

os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(model, f"{MODEL_DIR}/{MODEL_FILE}")
print("Model saved locally")

Model saved locally


In [39]:
blob = bucket.blob("dev/model/model.joblib")
blob.upload_from_filename(f"{MODEL_DIR}/{MODEL_FILE}")
print("Model uploaded to GCS")

Model uploaded to GCS


In [40]:
from sklearn.metrics import f1_score

y_pred = model.predict(X)
f1 = f1_score(y, y_pred, average="weighted")
f1

0.9706573727285337

In [41]:
F1_THRESHOLD = 0.85
DEPLOY_DECISION = f1 >= F1_THRESHOLD
print("Deploy?", DEPLOY_DECISION)

Deploy? True


In [42]:
uploaded_model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    artifact_uri=f"gs://{BUCKET_NAME}/dev/model",
    serving_container_image_uri="europe-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest",
)

Creating Model
Create Model backing LRO: projects/71707089683/locations/europe-west3/models/531059718169296896/operations/7490487010776317952
Model created. Resource name: projects/71707089683/locations/europe-west3/models/531059718169296896@1
To use this Model in another session:
model = aiplatform.Model('projects/71707089683/locations/europe-west3/models/531059718169296896@1')


In [29]:
endpoints = aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_NAME}"')

if endpoints:
    endpoint = endpoints[0]
    print("Using existing endpoint")
else:
    endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_NAME)
    print("Created new endpoint")

endpoint

Creating Endpoint
Create Endpoint backing LRO: projects/71707089683/locations/europe-west1/endpoints/3605010555431026688/operations/7201335656197390336
Endpoint created. Resource name: projects/71707089683/locations/europe-west1/endpoints/3605010555431026688
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/71707089683/locations/europe-west1/endpoints/3605010555431026688')
Created new endpoint


resource name: projects/71707089683/locations/europe-west1/endpoints/3605010555431026688

In [ ]:
endpoint.deploy(
    model=uploaded_model,
    deployed_model_display_name="knn-dev-v1",
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=2,
)


Deploying Model projects/71707089683/locations/europe-west1/models/5070809088837812224 to Endpoint : projects/71707089683/locations/europe-west1/endpoints/3605010555431026688
Deploy Endpoint model backing LRO: projects/71707089683/locations/europe-west1/endpoints/3605010555431026688/operations/5032852435618496512


FailedPrecondition: 400 Model server never became ready. Please validate that your model file or container configuration are valid. Model server logs can be found at https://console.cloud.google.com/logs/viewer?project=71707089683&resource=aiplatform.googleapis.com%2FEndpoint&advancedFilter=resource.type%3D%22aiplatform.googleapis.com%2FEndpoint%22%0Aresource.labels.endpoint_id%3D%223605010555431026688%22%0Aresource.labels.location%3D%22europe-west1%22. 9: Model server never became ready. Please validate that your model file or container configuration are valid. Model server logs can be found at https://console.cloud.google.com/logs/viewer?project=71707089683&resource=aiplatform.googleapis.com%2FEndpoint&advancedFilter=resource.type%3D%22aiplatform.googleapis.com%2FEndpoint%22%0Aresource.labels.endpoint_id%3D%223605010555431026688%22%0Aresource.labels.location%3D%22europe-west1%22.

In [ ]:
endpoint.predict(instances=[X.iloc[0].tolist()])